In [16]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/MSc Data Science/Dissertation/Exploration_Examples')

# Libraries
import numpy as np
import pandas as pd
import data_gens as dg
import evals
import models

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Synthetic Example 1: Pairwise Dataset Evaluation with BT, MM and GMM


## Model-Based N!

In [17]:
# Generate the synthetic dataset
df_matches, df_rank_gt, str_dic = dg.bt_syntheticdata_gen(n_items=3, n_matches=50, random_seed=23)

# Extract the standard items list from the ground truth dataframe to feed into the models
items = df_rank_gt['Item'].tolist()

# Ranking by Wins
# Count wins and merge with the full items list (in case there are items with 0 wins)
win_counts = df_matches['Winner'].value_counts().reset_index()
win_counts.columns = ['Item', 'Wins']
df_wins = pd.DataFrame({'Item': items}).merge(win_counts, on='Item', how='left').fillna(0)
df_wins['Wins'] = df_wins['Wins'].astype(int)

# Sort by Wins and filter columns
df_wins = df_wins.sort_values(by='Wins', ascending=False).reset_index(drop=True)

print("Ranking by Wins")
print(df_wins[['Item', 'Wins']])
print("\n")

# Ranking by Ground Truth (Betas)
# Create the Rank column based on the GT Beta values
df_rank_gt['Rank'] = df_rank_gt['GT Beta'].rank(ascending=False, method='min').astype(int)

# Sort by Rank and filter columns
df_gt_sorted = df_rank_gt.sort_values(by='Rank', ascending=True).reset_index(drop=True)

print("Ranking by Ground Truth (Betas)")
print(df_gt_sorted[['Item', 'Rank', 'GT Beta']])

Ranking by Wins
     Item  Wins
0  Item_1    33
1  Item_2    30
2  Item_3    12


Ranking by Ground Truth (Betas)
     Item  Rank   GT Beta
0  Item_1     1  0.666988
1  Item_2     2  0.025813
2  Item_3     3 -0.777619


In [18]:
# Fit Bradley-Terry Model
print("Bradley-Terry Model")
df_bt = models.fit_bt_pw(df_matches, items)
print(df_bt)
print("\n")

# Fit Mallows Model
print("Mallows Model")
df_mm, min_penalty = models.fit_mm_pw(df_matches, items)
print(f"Penalty: {min_penalty}\n")
print(df_mm)
print("\n")

# Fit Generalized Mallows Model (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained, theta_j_unconstrained = models.fit_gmm_pw(df_matches, items, constraint=None)
print(df_gmm_unconstrained)
print("\n")

# Fit Generalized Mallows Model (Constrained)
print("Generalized Mallows Model (Constrained)")
df_gmm_constrained, theta_j_constrained = models.fit_gmm_pw(df_matches, items, constraint='decreasing')
print(df_gmm_constrained)

Bradley-Terry Model
     Item      Beta
0  Item_1  0.473067
1  Item_2  0.296710
2  Item_3 -0.769777


Mallows Model
Penalty: 24.0

     Item
0  Item_1
1  Item_2
2  Item_3


Generalized Mallows Model (Unconstrained)
Penalties (Unconstrained)
     Item
0  Item_1
1  Item_2
2  Item_3


Generalized Mallows Model (Constrained)
Penalties (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3


In [19]:
# Evaluation of Pairwise Data Model's Performance

# Extract the true ranking
true_ranking = df_gt_sorted['Item'].tolist()

# Models' predicted rankings dictionary
model_rankings = {'BT': df_bt['Item'].tolist(),
                  'MM': df_mm['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained['Item'].tolist()}

# Calculate the optimal theta penalties according to the ground truth centre
theta_j_gt = evals.calculate_gt_penalties(df_matches, items, true_ranking)

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)
print("\n")

# Positional model's performance
print("Positional Model Performance")
gmm_vectors, df_gmm_positional = evals.evaluate_gmm_positional_penalties(true_ranking, model_rankings, theta_j_gt)

Overall Model Performance
BT: 0
MM: 0
GMM (U): 0
GMM (C): 0


Positional Model Performance
         Rank_1  Rank_2  Rank_3
BT          0.0     0.0     0.0
MM          0.0     0.0     0.0
GMM (U)     0.0     0.0     0.0
GMM (C)     0.0     0.0     0.0


## Sampling-Based Inference

### Markov Chain Monte Carlo (MCMC) - Metropolis Hastings

In [20]:
np.random.seed(23)

# Fit Bradley-Terry MCMC Model
print("Bradley-Terry Model")
df_bt_mcmc = models.fit_bt_pw(df_matches, items, stat_mthd='MCMC', iterations=1000, proposal_type='normal', step_size=0.1, burn_in=None)
print(df_bt_mcmc.head(10))
print("\n")

# Fit Mallows Model MCMC
print("Mallows Model")
df_mm_mcmc, min_penalty_mcmc = models.fit_mm_pw(df_matches, items, stat_mthd='MCMC', iterations=1000, proposal_type='swap', burn_in=None)
print(f"Penalty: {min_penalty_mcmc}\n")
print(df_mm_mcmc.head(10))
print("\n")

# Fit Generalized Mallows Model MCMC (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained_mcmc, theta_j_unconstrained_mcmc = models.fit_gmm_pw(df_matches, items, constraint=None, stat_mthd='MCMC', iterations=1000, proposal_type='normal', step_size=0.1, burn_in=None)
print(df_gmm_unconstrained_mcmc.head(10))
print("\n")

# Fit Generalized Mallows Model MCMC (Constrained)
print("Generalized Mallows Model (Constrained)")
df_gmm_constrained_mcmc, theta_j_constrained_mcmc = models.fit_gmm_pw(df_matches, items, constraint='decreasing', stat_mthd='MCMC', iterations=1000, proposal_type='normal', step_size=0.1, burn_in=None)
print(df_gmm_constrained_mcmc.head(10))

Bradley-Terry Model
     Item      Beta
0  Item_1  0.387065
1  Item_2  0.359954
2  Item_3 -0.747019


Mallows Model
Penalty: 24.0

     Item
0  Item_1
1  Item_2
2  Item_3


Generalized Mallows Model (Unconstrained)
Penalties MCMC (Unconstrained)
     Item
0  Item_2
1  Item_1
2  Item_3


Generalized Mallows Model (Constrained)
Penalties MCMC (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3


In [21]:
# Evaluation of Pairwise Data Model's Performance with MCMC

# Extract the true ranking
true_ranking = df_gt_sorted['Item'].tolist()

# Models' predicted rankings dictionary
model_rankings = {'BT': df_bt_mcmc['Item'].tolist(),
                  'MM': df_mm_mcmc['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained_mcmc['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained_mcmc['Item'].tolist()}

# Calculate the optimal theta penalties according to the ground truth centre
theta_j_gt = evals.calculate_gt_penalties(df_matches, items, true_ranking)

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)
print("\n")

# Positional model's performance
print("Positional Model Performance")
gmm_vectors, df_gmm_positional = evals.evaluate_gmm_positional_penalties(true_ranking, model_rankings, theta_j_gt)

Overall Model Performance
BT: 0
MM: 0
GMM (U): 1
GMM (C): 0


Positional Model Performance
         Rank_1  Rank_2  Rank_3
BT        0.000     0.0     0.0
MM        0.000     0.0     0.0
GMM (U)   0.001     0.0     0.0
GMM (C)   0.000     0.0     0.0


### Sequential Monte Carlo (SMC) - Adaptive Free-Tuning

In [22]:
np.random.seed(23)

# Fit Bradley-Terry SMC Model
print("Bradley-Terry Model")
df_bt_smc = models.fit_bt_pw(df_matches, items, stat_mthd='SMC', iterations=1000, burn_in=None)
print(df_bt_smc.head(10))
print("\n")

# Fit Mallows Model SMC
print("Mallows Model")
df_mm_smc, min_penalty_smc = models.fit_mm_pw(df_matches, items, stat_mthd='SMC', iterations=1000, burn_in=None)
print(f"Penalty: {min_penalty_smc}\n")
print(df_mm_smc.head(10))
print("\n")

# Fit Generalized Mallows Model SMC (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained_smc, theta_j_unconstrained_smc = models.fit_gmm_pw(df_matches, items, constraint=None, stat_mthd='SMC', iterations=1000, burn_in=None)
print(df_gmm_unconstrained_smc.head(10))
print("\n")

# Fit Generalized Mallows Model SMC (Constrained)
print("Generalized Mallows Model (Constrained)")
df_gmm_constrained_smc, theta_j_constrained_smc = models.fit_gmm_pw(df_matches, items, constraint='decreasing', stat_mthd='SMC', iterations=1000, burn_in=None)
print(df_gmm_constrained_smc.head(10))

Bradley-Terry Model
     Item      Beta
0  Item_1  0.558763
1  Item_2  0.302988
2  Item_3 -0.861752


Mallows Model
Penalty: 24.0

     Item
0  Item_1
1  Item_2
2  Item_3


Generalized Mallows Model (Unconstrained)
Penalties SMC (Unconstrained)
     Item
0  Item_1
1  Item_2
2  Item_3


Generalized Mallows Model (Constrained)
Penalties SMC (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3


In [23]:
# Evaluation of Pairwise Data Model's Performance with SMC

# Extract the true ranking
true_ranking = df_gt_sorted['Item'].tolist()

# Models' predicted rankings dictionary
model_rankings = {'BT': df_bt_smc['Item'].tolist(),
                  'MM': df_mm_smc['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained_smc['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained_smc['Item'].tolist()}

# Calculate the optimal theta penalties according to the ground truth centre
theta_j_gt = evals.calculate_gt_penalties(df_matches, items, true_ranking)

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)
print("\n")

# Positional model's performance
print("Positional Model Performance")
gmm_vectors, df_gmm_positional = evals.evaluate_gmm_positional_penalties(true_ranking, model_rankings, theta_j_gt)

Overall Model Performance
BT: 0
MM: 0
GMM (U): 0
GMM (C): 0


Positional Model Performance
         Rank_1  Rank_2  Rank_3
BT          0.0     0.0     0.0
MM          0.0     0.0     0.0
GMM (U)     0.0     0.0     0.0
GMM (C)     0.0     0.0     0.0


# Synthetic Example 2: Mallows-Type (Full Rankings) Dataset Evaluation with MM and GMM

## Model-Based N!

In [24]:
# Generate the synthetic dataset
df_gmm, gt_gmm = dg.mallows_syntheticdata_gen(n_items=3, n_judges=50, model_type='GMM', true_references=None, true_thetas=None, random_seed=23)

# Extract the standard items list from the ground truth dataframe to feed into the models
rank_cols = [col for col in df_gmm.columns if col.startswith('Rank_')]
items = df_gmm.iloc[0][rank_cols].tolist()

print(df_gmm)
print(f"Ground Truth Consensus: {gt_gmm['Clusters'][0]['Consensus']}")
print(f"Ground Truth Thetas: {np.round(gt_gmm['Clusters'][0]['Theta'], 4)}\n")

    Judge_ID  Cluster  Rank_1  Rank_2  Rank_3
0          1        0  Item_1  Item_2  Item_3
1          2        0  Item_1  Item_2  Item_3
2          3        0  Item_3  Item_1  Item_2
3          4        0  Item_3  Item_1  Item_2
4          5        0  Item_1  Item_2  Item_3
5          6        0  Item_1  Item_2  Item_3
6          7        0  Item_1  Item_2  Item_3
7          8        0  Item_1  Item_3  Item_2
8          9        0  Item_1  Item_2  Item_3
9         10        0  Item_1  Item_3  Item_2
10        11        0  Item_1  Item_3  Item_2
11        12        0  Item_1  Item_2  Item_3
12        13        0  Item_2  Item_3  Item_1
13        14        0  Item_1  Item_2  Item_3
14        15        0  Item_1  Item_2  Item_3
15        16        0  Item_2  Item_1  Item_3
16        17        0  Item_1  Item_3  Item_2
17        18        0  Item_1  Item_3  Item_2
18        19        0  Item_1  Item_3  Item_2
19        20        0  Item_3  Item_1  Item_2
20        21        0  Item_1  Ite

In [25]:
# Fit Mallows Model
print("Mallows Model")
df_mm, min_penalty_mm = models.fit_mm_full(df_gmm, items)
print(df_mm)
print(f"Minimum Penalty: {min_penalty_mm}")
print("\n")


# Fit Generalized Mallows Model (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained, thetas_unconstrained = models.fit_gmm_full(df_gmm, items, constraint=None)
print(df_gmm_unconstrained)
print(f"Inferred Thetas: {np.round(thetas_unconstrained, 4)}")
print("\n")


# Fit Generalized Mallows Model (Constrained)
print("Generalized Mallows Model (Constrained) ")
df_gmm_constrained, thetas_constrained = models.fit_gmm_full(df_gmm, items, constraint='decreasing')
print(df_gmm_constrained)
print(f"Inferred Thetas: {np.round(thetas_constrained, 4)}")

Mallows Model
     Item
0  Item_1
1  Item_2
2  Item_3
Minimum Penalty: 41.0


Generalized Mallows Model (Unconstrained)
Penalties (Unconstrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [0.001 0.001]


Generalized Mallows Model (Constrained) 
Penalties (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [0.001 0.001]


In [26]:
# Evaluation of Full Rankings Model's Performance

# Extract the true ranking
true_ranking = gt_gmm['Clusters'][0]['Consensus']

# Models' predicted rankings dictionary
model_rankings = {'MM': df_mm['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained['Item'].tolist()}

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)

Overall Model Performance
MM: 0
GMM (U): 0
GMM (C): 0


## Sampling-Based Inference

### Markov Chain Monte Carlo (MCMC) - Metropolis Hastings

In [27]:
np.random.seed(23)

# Fit Mallows Model
print("Mallows Model")
df_mm_mcmc, min_penalty_mm = models.fit_mm_full(df_gmm, items, stat_mthd='MCMC', iterations=1000, proposal_type="swap", burn_in=None)
print(df_mm_mcmc)
print(f"Minimum Penalty: {min_penalty_mm}")
print("\n")


# Fit Generalized Mallows Model (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained_mcmc, thetas_unconstrained_mcmc = models.fit_gmm_full(df_gmm, items, constraint=None, stat_mthd='MCMC', iterations=1000, proposal_type="normal", step_size=0.001, burn_in=None)
print(df_gmm_unconstrained_mcmc)
print(f"Inferred Thetas: {np.round(thetas_unconstrained_mcmc, 4)}")
print("\n")


# Fit Generalized Mallows Model (Constrained)
print("Generalized Mallows Model (Constrained) ")
df_gmm_constrained_mcmc, thetas_constrained_mcmc = models.fit_gmm_full(df_gmm, items, constraint='decreasing', stat_mthd='MCMC', iterations=1000, proposal_type="normal", step_size=0.001, burn_in=None)
print(df_gmm_constrained_mcmc)
print(f"Inferred Thetas: {np.round(thetas_constrained_mcmc, 4)}")

Mallows Model
     Item
0  Item_1
1  Item_2
2  Item_3
Minimum Penalty: 41.0


Generalized Mallows Model (Unconstrained)
Penalties MCMC (Unconstrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [1.0132 0.9794]


Generalized Mallows Model (Constrained) 
Penalties MCMC (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [1.9831 0.5219]


In [28]:
# Evaluation of Full Rankings Model's Performance with MCMC

# Extract the true ranking
true_ranking = gt_gmm['Clusters'][0]['Consensus']

# Models' predicted rankings dictionary
model_rankings = {'MM': df_mm_mcmc['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained_mcmc['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained_mcmc['Item'].tolist()}

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)

Overall Model Performance
MM: 0
GMM (U): 0
GMM (C): 0


### Sequential Monte Carlo (SMC) - Adaptive Free-Tuning

In [29]:
np.random.seed(23)

# Fit Mallows Model
print("Mallows Model")
df_mm_smc, min_penalty_mm = models.fit_mm_full(df_gmm, items, stat_mthd='SMC', iterations=1000)
print(df_mm_smc)
print(f"Minimum Penalty: {min_penalty_mm}")
print("\n")


# Fit Generalized Mallows Model (Unconstrained)
print("Generalized Mallows Model (Unconstrained)")
df_gmm_unconstrained_smc, thetas_unconstrained_smc = models.fit_gmm_full(df_gmm, items, constraint=None, stat_mthd='SMC', iterations=1000)
print(df_gmm_unconstrained_smc)
print(f"Inferred Thetas: {np.round(thetas_unconstrained_smc, 4)}")
print("\n")


# Fit Generalized Mallows Model (Constrained)
print("Generalized Mallows Model (Constrained) ")
df_gmm_constrained_smc, thetas_constrained_smc = models.fit_gmm_full(df_gmm, items, constraint='decreasing', stat_mthd='SMC', iterations=1000)
print(df_gmm_constrained_smc)
print(f"Inferred Thetas: {np.round(thetas_constrained_smc, 4)}")

Mallows Model
Optimal Theta Selected by SMC: 0.5788
     Item
0  Item_1
1  Item_2
2  Item_3
Minimum Penalty: 41.0


Generalized Mallows Model (Unconstrained)
Penalties SMC (Unconstrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [0.9958 0.9965]


Generalized Mallows Model (Constrained) 
Penalties SMC (Constrained)
     Item
0  Item_1
1  Item_2
2  Item_3
Inferred Thetas: [1.9498 0.4803]


In [30]:
# Evaluation of Full Rankings Model's Performance with SMC

# Extract the true ranking
true_ranking = gt_gmm['Clusters'][0]['Consensus']

# Models' predicted rankings dictionary
model_rankings = {'MM': df_mm_smc['Item'].tolist(),
                  'GMM (U)': df_gmm_unconstrained_smc['Item'].tolist(),
                  'GMM (C)': df_gmm_constrained_smc['Item'].tolist()}

# Overall model's performance
print("Overall Model Performance")
mm_totals = evals.evaluate_mm_penalty(true_ranking, model_rankings)

Overall Model Performance
MM: 0
GMM (U): 0
GMM (C): 0
